# Salary Prediction — Data Preprocessing
## Overview
This notebook applies all preprocessing decisions from EDA:
- Drop irrelevant columns
- Handle outliers via Winsorization
- Feature engineering
- Encode categorical variables
- Save processed dataset

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/ds_salaries.csv')
print(df.head())

   Unnamed: 0  work_year experience_level employment_type  \
0           0       2020               MI              FT   
1           1       2020               SE              FT   
2           2       2020               SE              FT   
3           3       2020               MI              FT   
4           4       2020               SE              FT   

                    job_title  salary salary_currency  salary_in_usd  \
0              Data Scientist   70000             EUR          79833   
1  Machine Learning Scientist  260000             USD         260000   
2           Big Data Engineer   85000             GBP         109024   
3        Product Data Analyst   20000             USD          20000   
4   Machine Learning Engineer  150000             USD         150000   

  employee_residence  remote_ratio company_location company_size  
0                 DE             0               DE            L  
1                 JP             0               JP            S  

In [2]:
df.drop(columns= ['Unnamed: 0', 'salary', 'salary_currency', 'employment_type', 'employee_residence'], inplace=True)

In [3]:
print(df.columns)
print(df.shape)

Index(['work_year', 'experience_level', 'job_title', 'salary_in_usd',
       'remote_ratio', 'company_location', 'company_size'],
      dtype='str')
(607, 7)


In [4]:
df['salary_in_usd'] = df['salary_in_usd'].clip(upper=280911)

In [5]:
print(df['salary_in_usd'].max())
print(df['salary_in_usd'].describe())

280911
count       607.000000
mean     110031.164745
std       62145.687298
min        2859.000000
25%       62726.000000
50%      101570.000000
75%      150000.000000
max      280911.000000
Name: salary_in_usd, dtype: float64


In [6]:
title_counts = df['job_title'].value_counts()
print('Titles with 10+ records:', (title_counts >= 10).sum())
print('Titles grouped as others:', (title_counts < 10).sum())
print("\nTitles kept: ")
print(title_counts[title_counts >= 10])

top_titles = title_counts[title_counts>=10].index
df['job_title'].isin(top_titles)
df['job_title'] = df['job_title'].where(df['job_title'].isin(top_titles), other='Other')
print(df['job_title'].value_counts())

Titles with 10+ records: 7
Titles grouped as others: 43

Titles kept: 
job_title
Data Scientist               143
Data Engineer                132
Data Analyst                  97
Machine Learning Engineer     41
Research Scientist            16
Data Science Manager          12
Data Architect                11
Name: count, dtype: int64
job_title
Other                        155
Data Scientist               143
Data Engineer                132
Data Analyst                  97
Machine Learning Engineer     41
Research Scientist            16
Data Science Manager          12
Data Architect                11
Name: count, dtype: int64


In [7]:
high_income = ['US', 'GB', 'CA', 'DE', 'AU', 'NL', 'SE', 'CH', 'DK', 'NO', 'NZ', 'FR', 'IE', 'AT', 'BE', 'SG']
middle_income = ['IN', 'BR', 'MX', 'PL', 'PT', 'ES', 'GR', 'RU', 'TR', 'CN', 'UA', 'RO', 'HR', 'HU', 'CZ']

# Method 1: Using np.select() it's the industry standard for multiple condition mapping.
import numpy as np
conditions = [
    df['company_location'].isin(high_income),
    df['company_location'].isin(middle_income)
]
choices = ['High', 'Middle']

df['company_location'] = np.select(conditions, choices, default='Low')
print(df['company_location'].value_counts())

company_location
High      496
Middle     76
Low        35
Name: count, dtype: int64


 Method 2: Using apply() with a custom function
def map_location(country):
    if country in high_income:
        return "High"
    elif country in middle_income:
        return 'Middle'
    else:
        return "Low"

df["company_location"] = df['company_location'].apply(map_location)

Two main types of encoding:
Ordinal Encoding — used when categories have a natural order.
Example: experience_level has a clear progression:

Entry < Mid < Senior < Executive

Label Encoding / One-Hot Encoding — used when categories have NO natural order.
Example: job_title — is "Data Scientist" more than "Data Engineer"? No — they're just different. There's no ranking between them.

For these you either:

Assign random numbers (Label Encoding) — risky because the model might think 3 > 1 means something
Create separate binary columns (One-Hot Encoding) — safer but creates many columns

In [8]:
experience_map = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
df['experience_level'] = df['experience_level'].map(experience_map)
print(df['experience_level'].value_counts())

experience_level
2    280
1    213
0     88
3     26
Name: count, dtype: int64


In [9]:
company_map = {'S': 0, 'M': 1, 'L': 2}
df['company_size'] = df['company_size'].map(company_map)
print(df['company_size'].value_counts())

company_size
1    326
2    198
0     83
Name: count, dtype: int64


In [10]:
company_location_map = {'Low': 0, 'Middle': 1, 'High': 2}
df['company_location'] = df['company_location'].map(company_location_map)
print(df['company_location'].value_counts())

company_location
2    496
1     76
0     35
Name: count, dtype: int64


The case for One-Hot Encoding:

No false ordering — the model won't think "Data Engineer (2) is greater than Data Analyst (0)"
Better for algorithms that are sensitive to numeric ordering like linear regression

The case for Label Encoding with Decision Trees:

Decision Trees split data based on thresholds — they don't assume numeric ordering means anything
A Decision Tree will find "job_title == 3" as a split condition regardless of what 3 means
One-Hot Encoding creates 8 new columns (one per title) — more complexity, more memory

The key question: What model are you training?
You're training a Decision Tree. Decision Trees handle label encoded categories perfectly well because they don't interpret the magnitude of numbers — they just find the best split point.
One-Hot Encoding is more important for:

Linear Regression
Logistic Regression
Neural Networks
Any model that treats features as continuous numbers

Conclusion: For a Decision Tree, Label Encoding is fine and more efficient. One-Hot Encoding would add unnecessary complexity without improving performance.


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['job_title'] = le.fit_transform(df['job_title'])
print(df['job_title'].value_counts())

#fit() — learns the unique categories in your data.
#It looks at all unique values in job_title and creates an internal mapping:
#transform() — applies that mapping to replace the actual values.
#It goes through every row and replaces the text with the corresponding number.

job_title
6    155
4    143
2    132
0     97
5     41
7     16
3     12
1     11
Name: count, dtype: int64


In [13]:
print(df.head())
print(df.dtypes)
print(df.shape)

   work_year  experience_level  job_title  salary_in_usd  remote_ratio  \
0       2020                 1          4          79833             0   
1       2020                 2          6         260000             0   
2       2020                 2          6         109024            50   
3       2020                 1          6          20000             0   
4       2020                 2          5         150000            50   

   company_location  company_size  
0                 2             2  
1                 0             0  
2                 2             1  
3                 0             0  
4                 2             2  
work_year           int64
experience_level    int64
job_title           int64
salary_in_usd       int64
remote_ratio        int64
company_location    int64
company_size        int64
dtype: object
(607, 7)


In [14]:
df.to_csv('../data/processed/processed_salaries.csv', index=False)